# 🗣️ Project Ē-Kóng (會講) — Colab 啟動器

**依序執行每個 Cell**（首次跑約需 5–8 分鐘安裝依賴）

| Step | 說明 |
|------|------|
| 1 | 確認 GPU / 環境 |
| 2 | Clone / 更新程式碼 |
| 3 | 設定 Secrets (.env) |
| 4 | 執行啟動腳本 |


In [ ]:
# ── Cell 1: 確認 GPU 環境 ─────────────────────────────────────────────────────
import subprocess

result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.free',
                         '--format=csv,noheader'], capture_output=True, text=True)
print('GPU 資訊：', result.stdout.strip() or '⚠️  未偵測到 GPU（請確認 Runtime > Change runtime type > T4 GPU）')

import sys
print(f'Python {sys.version}')

In [ ]:
# ── Cell 2: Clone 或更新程式碼 ────────────────────────────────────────────────
import os
from pathlib import Path

REPO_URL = 'https://github.com/YOUR_USERNAME/E-Kong.git'  # ← 替換成你的 repo URL
WORK_DIR = Path('/content/E-Kong')

if WORK_DIR.exists():
    print('🔄 更新程式碼...')
    os.system(f'git -C {WORK_DIR} pull')
else:
    print('📥 Clone 程式碼...')
    os.system(f'git clone {REPO_URL} {WORK_DIR}')

os.chdir(WORK_DIR)
print(f'✅ 工作目錄：{os.getcwd()}')

In [ ]:
# ── Cell 3: 設定 .env（使用 Colab Secrets 或直接填入）─────────────────────────
# 方法 A（推薦）：使用 Colab Secrets（左側鑰匙圖示加入 key）
from google.colab import userdata

env_content = f"""LINE_CHANNEL_ACCESS_TOKEN={userdata.get('LINE_CHANNEL_ACCESS_TOKEN')}
LINE_CHANNEL_SECRET={userdata.get('LINE_CHANNEL_SECRET')}
NGROK_AUTH_TOKEN={userdata.get('NGROK_AUTH_TOKEN')}
APP_PORT=8000
LOG_LEVEL=INFO
LLM_MODEL_PATH=/content/drive/MyDrive/ekong_models/taide-lx-7b-chat.Q4_K_M.gguf
LLM_N_GPU_LAYERS=35
LLM_MAX_TOKENS=512
LLM_TEMPERATURE=0.7
WHISPER_MODEL_SIZE=base
WHISPER_DEVICE=cuda
WHISPER_COMPUTE_TYPE=int8
"""

with open('/content/E-Kong/.env', 'w') as f:
    f.write(env_content)

print('✅ .env 已建立（內容已隱藏）')

In [ ]:
# ── Cell 4: 執行啟動腳本（安裝依賴 + 啟動 ngrok + 啟動 FastAPI）────────────────
# 若只是重啟伺服器（不需重裝套件），可直接跳到 Cell 5
%cd /content/E-Kong
!python setup_colab.py

In [ ]:
# ── Cell 5: 快速重啟（已安裝依賴，跳過安裝直接起服務）──────────────────────────
import os
import subprocess
import time
from dotenv import load_dotenv
from pyngrok import conf as ngrok_conf, ngrok
import httpx

os.chdir('/content/E-Kong')
load_dotenv()

APP_PORT = int(os.getenv('APP_PORT', '8000'))
ngrok_conf.get_default().auth_token = os.environ['NGROK_AUTH_TOKEN']

# 終止舊 tunnel
ngrok.kill()
tunnel = ngrok.connect(APP_PORT, bind_tls=True)
public_url = tunnel.public_url.rstrip('/')

# 更新 LINE Webhook
with httpx.Client(timeout=15) as client:
    resp = client.put(
        'https://api.line.me/v2/bot/channel/webhook/endpoint',
        headers={'Authorization': f"Bearer {os.environ['LINE_CHANNEL_ACCESS_TOKEN']}",
                 'Content-Type': 'application/json'},
        json={'webhook': f'{public_url}/webhook'},
    )
print(f'LINE Webhook 更新: {resp.status_code}')

# 啟動 FastAPI
proc = subprocess.Popen(['python', '-m', 'uvicorn', 'app.main:app',
                          '--host', '0.0.0.0', '--port', str(APP_PORT)])
time.sleep(3)
print(f'🎉 服務已啟動！')
print(f'   Webhook : {public_url}/webhook')
print(f'   Health  : {public_url}/health')
print(f'   Docs    : {public_url}/docs')